In [1]:
import requests
import requests
import pandas as pd
import time
import logging
from pathlib import Path
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

API_KEY  = "HDEV-907006be-6b96-44d1-bff6-ade257f217ff"
BASE_URL = "https://api.henrikdev.xyz/valorant"
HEADERS  = {"Authorization": API_KEY}

RATE_SLEEP         = 3.0   # 30 req/min
MATCHES_PER_PLAYER = 5     # v3 最多支持 size=10，每人拉 5 场足够
MAX_MATCHES        = 1500  # 全局 unique match 上限
SAMPLE_PER_REGION  = 200
OUT_DIR            = Path("data")
OUT_DIR.mkdir(exist_ok=True)

#### 1. Stratified Sampling

In [2]:
def stratified_sample(pool_path: str = "players_pool.csv") -> pd.DataFrame:
    df      = pd.read_csv(pool_path)
    sampled = (
        df.groupby("region", group_keys=False)
          .apply(lambda g: g.sample(n=min(SAMPLE_PER_REGION, len(g)), random_state=42),
                 include_groups=False)
          .reset_index(drop=True)
    )
    # include_groups=False 避免 FutureWarning，但会丢失 region 列，需要 merge 回来
    sampled = df.merge(sampled, how="inner").drop_duplicates().reset_index(drop=True)
    log.info(f"Sampled {len(sampled)} players")
    log.info(sampled["region"].value_counts().to_string())
    sampled.to_csv(OUT_DIR / "sampled_players.csv", index=False)
    return sampled

#### 2. Basic Packing for Requesting

In [3]:
def get(url: str, params: dict = None, retries: int = 3) -> dict | None:
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:
                wait = 30 * (attempt + 1)
                log.warning(f"Rate limited (429), waiting {wait}s …")
                time.sleep(wait)
            elif r.status_code == 404:
                return None
            else:
                log.warning(f"HTTP {r.status_code}: {url}")
                return None
        except Exception as e:
            log.error(f"Request error ({attempt+1}/{retries}): {e}")
            time.sleep(5)
    return None


def rate_limited_get(url, params=None):
    result = get(url, params)
    time.sleep(RATE_SLEEP)
    return result

#### 3. Matchlist


In [4]:
def fetch_matches(puuid: str, region: str, size: int = MATCHES_PER_PLAYER) -> list[dict]:
    url  = f"{BASE_URL}/v3/by-puuid/matches/{region}/{puuid}"
    data = rate_limited_get(url, params={"mode": "competitive", "size": size})
    if not data or str(data.get("status")) != "200":
        return []
    return data.get("data", [])

#### 4. Decompose Matchlist

In [5]:
def parse_match(match: dict) -> list[dict]:
    meta     = match.get("metadata", {})
    match_id = meta.get("matchid")

    # v3 teams 是 dict: {red: {has_won, ...}, blue: {has_won, ...}}
    teams    = match.get("teams", {})
    team_won = {
        "Red":  teams.get("red",  {}).get("has_won"),
        "Blue": teams.get("blue", {}).get("has_won"),
    }

    rows = []
    for p in match.get("players", {}).get("all_players", []):
        s  = p.get("stats", {})
        ec = p.get("economy", {})
        bh = p.get("behavior", {})
        ab = p.get("ability_casts", {})

        total_shots = (s.get("headshots", 0) + s.get("bodyshots", 0) + s.get("legshots", 0)) or 1
        team        = p.get("team")

        rows.append({
            # 识别
            "match_id":   match_id,
            "puuid":      p.get("puuid"),
            "name":       p.get("name"),
            "tag":        p.get("tag"),
            "team":       team,
            "region":     meta.get("region"),
            # 控制变量
            "map":           meta.get("map"),
            "patch":         meta.get("game_version"),
            "rounds_played": meta.get("rounds_played"),
            "game_start":    meta.get("game_start"),
            "rank_tier":     p.get("currenttier"),
            "rank_patched":  p.get("currenttier_patched"),
            "agent":         p.get("character"),
            "player_level":  p.get("level"),
            # 绩效
            "score":           s.get("score"),
            "kills":           s.get("kills"),
            "deaths":          s.get("deaths"),
            "assists":         s.get("assists"),
            "damage_made":     p.get("damage_made"),      # v3: 顶层字段
            "damage_received": p.get("damage_received"),
            "headshot_rate":   round(s.get("headshots", 0) / total_shots, 4),
            "headshots":       s.get("headshots"),
            "bodyshots":       s.get("bodyshots"),
            "legshots":        s.get("legshots"),
            # 能力使用（v3: c/q/e/x_cast）
            "c_cast": ab.get("c_cast", 0),
            "q_cast": ab.get("q_cast", 0),
            "e_cast": ab.get("e_cast", 0),
            "x_cast": ab.get("x_cast", 0),
            # 资源分配
            "economy_spent_overall": ec.get("spent",         {}).get("overall"),
            "economy_spent_avg":     ec.get("spent",         {}).get("average"),
            "loadout_value_overall": ec.get("loadout_value", {}).get("overall"),
            "loadout_value_avg":     ec.get("loadout_value", {}).get("average"),
            # 社会懈怠代理（v3: behaviour）
            "afk_rounds":      bh.get("afk_rounds"),
            "rounds_in_spawn": bh.get("rounds_in_spawn"),
            "ff_outgoing":     bh.get("friendly_fire", {}).get("outgoing"),
            "ff_incoming":     bh.get("friendly_fire", {}).get("incoming"),
            # 结果
            "team_won": team_won.get(team),
        })
    return rows

#### 5. Main Workflow

In [6]:
from soupsieve import match


def main(pool_path: str = "players_pool.csv", resume: bool = True):
    players = stratified_sample(pool_path)

    done_path = OUT_DIR / "done_match_ids.txt"
    done_ids: set[str] = set()
    if resume and done_path.exists():
        done_ids = set(done_path.read_text().splitlines())
        log.info(f"Resuming: {len(done_ids)} matches already done")

    out_path = OUT_DIR / "match_players.csv"
    buffer: list[dict] = []
    FLUSH_EVERY = 500

    # 每个地区均匀分配配额
    regions        = players["region"].unique()
    quota_per_region = MAX_MATCHES // len(regions)   # 1500 / 6 = 250
    region_counts: dict[str, int] = {r: 0 for r in regions}

    # 统计 resume 时已有的各地区数量（需要读已有数据）
    if resume and out_path.exists():
        existing = pd.read_csv(out_path, usecols=["match_id", "region"]).drop_duplicates("match_id")
        for r, cnt in existing["region"].value_counts().items():
            if r in region_counts:
                region_counts[r] = cnt
        log.info(f"Resume region counts: {region_counts}")

    # 按地区分组玩家
    players_by_region = {r: grp.to_dict("records") for r, grp in players.groupby("region")}
    region_player_idx = {r: 0 for r in regions}   # 每个地区当前遍历到第几个玩家

    total_done = sum(region_counts.values())

    with tqdm(total=MAX_MATCHES, desc="Unique matches", unit="match") as pbar:
        pbar.update(len(done_ids))

        while total_done < MAX_MATCHES:
            progressed = False

            for region in regions:
                if region_counts[region] >= quota_per_region:
                    continue

                idx = region_player_idx[region]
                pool = players_by_region[region]
                if idx >= len(pool):
                    log.warning(f"  [{region}] ran out of players before reaching quota")
                    continue

                row   = pool[idx]
                region_player_idx[region] += 1
                puuid = row["puuid"]

                matches = fetch_matches(puuid, region)
                for match in matches:
                    if not isinstance(match, dict) or not match.get("is_available", True):
                        continue
                    if match.get("metadata") is None:
                        continue
                    if region_counts[region] >= quota_per_region:
                        break
                    mid = match.get("metadata", {}).get("matchid")
                    if not mid or mid in done_ids:
                        continue

                    buffer.extend(parse_match(match))
                    done_ids.add(mid)
                    region_counts[region] += 1
                    total_done += 1
                    progressed  = True
                    pbar.update(1)
                    pbar.set_postfix({"region": region, **{r: region_counts[r] for r in regions}})

                    if len(buffer) >= FLUSH_EVERY:
                        _flush(buffer, done_ids, done_path, out_path)
                        buffer = []

            if not progressed:
                log.info("No progress made this round — all regions at quota or out of players.")
                break

    _flush(buffer, done_ids, done_path, out_path)
    log.info(f"✅ Done. Region counts: {region_counts}")


def _flush(rows, done_ids, done_path, out_path):
    if rows:
        pd.DataFrame(rows).to_csv(
            out_path, mode="a", header=not out_path.exists(), index=False
        )
        log.info(f"  Flushed {len(rows)} rows → {out_path}")
    done_path.write_text("\n".join(done_ids))

#### 6. Dry run

In [7]:
def dry_run(pool_path: str = "players_pool.csv",
            test_players: int = 2,
            test_matches: int = 3):
    log.info("=" * 50)
    log.info("DRY RUN START")
    log.info("=" * 50)

    out_dir = OUT_DIR / "dry_run"
    out_dir.mkdir(exist_ok=True)

    df_pool = pd.read_csv(pool_path)
    sample  = (
        df_pool.groupby("region", group_keys=False)
               .apply(lambda g: g.sample(n=min(test_players, len(g)), random_state=0),
                      include_groups=False)
               .reset_index(drop=True)
    )
    sample = df_pool.merge(sample, how="inner").drop_duplicates().reset_index(drop=True)
    log.info(f"Test sample: {len(sample)} players across {sample['region'].nunique()} regions")

    results, errors = [], []

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Dry run players"):
        puuid, region = row["puuid"], row["region"]

        matches = fetch_matches(puuid, region, size=test_matches)
        if not matches:
            errors.append({"puuid": puuid, "region": region, "step": "matchlist", "detail": "empty"})
            log.warning(f"  ✗ matchlist empty for {puuid[:8]} [{region}]")
            continue
        log.info(f"  ✓ matchlist OK — {puuid[:8]} [{region}] → {len(matches)} matches")

        rows = parse_match(matches[0])
        if not rows:
            errors.append({"puuid": puuid, "region": region, "step": "parse", "detail": "empty"})
            log.warning(f"  ✗ parse returned 0 rows")
            continue

        key_fields = ["score", "kills", "damage_made", "economy_spent_overall", "afk_rounds"]
        missing    = [f for f in key_fields if rows[0].get(f) is None]
        if missing:
            log.warning(f"  ⚠ missing fields {missing}")
        else:
            log.info(f"  ✓ parse OK — {len(rows)} players, all key fields present")

        results.extend(rows)

    if results:
        pd.DataFrame(results).to_csv(out_dir / "dry_run_output.csv", index=False)
        log.info(f"\nDry run output saved → {out_dir / 'dry_run_output.csv'}")

    log.info("\n" + "=" * 50)
    log.info("DRY RUN SUMMARY")
    log.info(f"  Players tested : {len(sample)}")
    log.info(f"  Rows collected : {len(results)}")
    log.info(f"  Errors         : {len(errors)}")
    if errors:
        for e in errors:
            log.info(f"    {e}")
    log.info("=" * 50)

    ok = len(errors) == 0
    if ok:
        log.info("✅ All checks passed — safe to run main()")
    else:
        log.warning("⚠ Fix errors above before running main()")
    return ok


#### 7. Processing

In [8]:
def build_concentration(df_path: str = "data/match_players.csv") -> pd.DataFrame:
    import numpy as np

    df = pd.read_csv(df_path).drop_duplicates(subset=["match_id", "puuid"])

    def gini(arr):
        arr = np.sort(np.abs(arr.values))
        n   = len(arr)
        if n == 0 or arr.sum() == 0:
            return np.nan
        idx = np.arange(1, n + 1)
        return ((2 * idx - n - 1) * arr).sum() / (n * arr.sum())

    grp = df.groupby(["match_id", "team"])
    df["score_share"] = df["score"] / grp["score"].transform("sum")
    df["gini_score"]  = grp["score"].transform(gini)
    df["top1_share"]  = grp["score"].transform("max") / grp["score"].transform("sum")
    df["is_carrier"]  = (df["score"] == grp["score"].transform("max")).astype(int)

    out = "data/match_players_enriched.csv"
    df.to_csv(out, index=False)
    log.info(f"Enriched dataset → {out}  shape={df.shape}")
    return df

In [9]:
import os
os.remove("data/match_players.csv")
os.remove("data/done_match_ids.txt")

In [10]:
if __name__ == "__main__":
    ok = True
    # ok = dry_run("players_pool.csv", test_players=2, test_matches=3)
    if ok:
        main("players_pool.csv")
        build_concentration()

2026-03-23 23:56:34,773 INFO Sampled 1200 players
2026-03-23 23:56:34,776 INFO region
kr       200
eu       200
latam    200
br       200
ap       200
na       200
Unique matches:   2%|▏         | 25/1500 [00:22<23:35,  1.04match/s, region=ap, kr=5, eu=5, latam=5, br=5, ap=5, na=0]   2026-03-23 23:56:57,417 WARNING Rate limited (429), waiting 30s …
2026-03-23 23:57:27,545 WARNING Rate limited (429), waiting 60s …
Unique matches:   4%|▎         | 54/1500 [02:17<42:32,  1.76s/match, region=ap, kr=10, eu=10, latam=10, br=10, ap=9, na=5]2026-03-23 23:58:52,574 WARNING Rate limited (429), waiting 30s …
2026-03-23 23:59:22,704 WARNING Rate limited (429), waiting 60s …
Unique matches:   7%|▋         | 108/1500 [05:21<39:13,  1.69s/match, region=br, kr=20, eu=20, latam=20, br=20, ap=14, na=14]   2026-03-24 00:01:57,411 WARNING Rate limited (429), waiting 30s …
2026-03-24 00:02:27,542 WARNING Rate limited (429), waiting 60s …
Unique matches:   9%|▊         | 131/1500 [07:21<56:28,  2.48s/match,